# **Measure Regions Morphology**

# <mark> TO DO LIST:
- <mark> check how volume fraction is handled when mask_name=None (morph base functions, 2.1, 2.2, and 2.4)
- <mark> update 2.1 to summarize organelle intensity value 

***Prior to this notebook, you should have already run through [2.0_quantification_setup](2.0_quantification_setup.ipynb) and have segmentation outputs of <ins>at least one</ins> region or mask (e.g., cell mask, nucleus, neurites, etc.).***

In notebooks 2.1 through 2.4, we will go over the implementation of `infer-subc` quantification methods (explained in detail in the `method_...` notebooks) to assess the morphology, interactions, and distribution of organelles at the single-cell level. 

### 📍 **Purpose**
This notebook can be used to measure the `morphology` -- the amount, size, and shape -- of one or more masks/regions within images. It includes options to:
1. 🦠 Quantify the morphology of *one or more mask(s)/region(s)* from <ins>ONE IMAGE</ins>
2. 🧪 Batch process the morphology of *one or more mask(s)/region(s)* from *multiple images* for a <ins>SINGLE EXPERIMENT</ins>
3. 🧮 Summarize morphology metrics *per image* across <INS>ONE OR MORE EXPERIMENTS</ins>

Below you will find an `explanation of steps` for these three quantitative methods. Then, more succinct code blocks are included to `execute quantification` on your own (or sample) data.

### 🍃 **Biological Relevance - Mask/region Morphology**
Outside of organelle-specific metrics (morphology, interactions, distribution), it can also be beneficial to understand more macro-level information about biological relevant sub-regions within your images. Some examples of this may be the whole or subcellular regions such as the nucleus/cytoplasm or neurites/soma.

In this analysis notebook, the morphology of masks/regions are measured using the methods described in [method_morphology.ipynb](method_morphology.ipynb). The following morphological measurements are included for each organelle:
- `label`: the unique ID number for the object being measured
- `centroid`: centroid coordinate tuple (row, col, Z)
- `bbox`: bounding box coordinates (min_row, min_col, max_row, max_col); pixels/voxels belonging to the bounding box are in the half-open interval [min_row; max_row) and [min_col; max_col).
- `area`: (or `volume` for 3D z-stack images) area of the region i.e. number of pixels of the region scaled by pixel-area; this metric has the option to be converted into "real world" units using the scale from the metadata.
- `surface_area`: the surface area of the region. For 3D, surface area of a 2D surface mesh of the region (skimage.measure.marching_cubes) using skimage.measure.mesh_surface_area; this metric has the option to be converted into "real world" units using the scale from the metadata.
- `SA_to_volume`: surface area / area (or volume); this metric has the option to be converted into "real world" units using the scale from the metadata.
- `equivalent_diameter`: the diameter of a circle with the same area as the region; this metric has the option to be converted into "real world" units using the scale from the metadata.
- `extent`: ratio of pixels/voxels in the region to pixels/voxels in the total bounding box. Computed as area / (rows * cols)
- `euler_number`: Euler characteristic of the set of non-zero pixels. Computed as number of connected components subtracted by number of holes (input.ndim connectivity). In 3D, number of connected components plus number of holes subtracted by number of tunnels.
- `solidity`: ratio of pixels/voxels in the region to pixels/voxels of the convex hull image.
- `axis_major_length`: the length of the major axis of the ellipse that has the same normalized second central moments as the region; this metric has the option to be converted into "real world" units using the scale from the metadata.
- `mask_volume`: the volume of the mask used to define the area of analysis; usually this will be the cell mask since the analysis is intended to be done at the single-cell level.

The following measures of the intensity images are also included:
- `min_intensity`: value with the least intensity in the region.
- `max_intensity`: value with the greatest intensity in the region.
- `mean_intensity`: value with the mean intensity in the region.
- `standard_deviation_intensity`: the standard deviation of the intensity in the region.


These measurements and definitions are derived from the [`skimage.measure.regionprops()`](https://scikit-image.org/docs/stable/api/skimage.measure.html#skimage.measure.regionprops) function. More in depth information about each measurement can be found there.

*You can learn more about the implementation of regionprops within infer-subc in the [method_morphology](method_morphology.ipynb) notebook.*

-----

## 🗂️ **Table of Contents** <mark> NEEDS UPDATING STILL
The following sections are included in this notebook:

**IMPORTS AND LOAD IMAGE**

**EXPLANATION OF STEPS** - This section serves as *expository examples* of the functions used to quantify, batch process, and summarize organelle morphology.

🦠 **Quantify the morphology of *one or more organelles* from <ins>ONE IMAGE</ins>**
- **`STEP 1`** - Select mask from regions list, if applicable
- **`STEP 2`** - Loop through the list of organelles to quantify the morphology of each
- **`STEP 3`** - Combine all of the tables together and add the image name as a metadata column
- **`DEFINE`** - The get_organelle_morph() function

🧪 **Batch process *multiple images* from a <ins>SINGLE EXPERIMENT</ins>**
- **`STEP 1`** - Check the output file paths to see if quantification data already exists
- **`STEP 2`** - List images and the segmentation file suffixes that should be collected for each
- **`STEP 3`** - Loop through the list of images and perform the morphology quantification on all organelles
- **`DEFINE`** - The batch_process_org_morph() function

🧮 **Summarize metrics *per image* across <INS>ONE OR MORE EXPERIMENTS</ins>**
- **`STEP 1`** - Get the orgnaelle morphology .csv files
- **`STEP 2`** - Summarize the mean, median, and standard deviation of each metric per image/mask region
- **`STEP 3`** - Ensure all organelles included in the analysis are represented and fill NA values with 0 as needed
- **`STEP 4`** - Unstack the organelle names and save file
- **`DEFINE`** - The batch_org_morph_summary_stats() function

**EXECUTE QUANTIFICATION** - Once you understand how the functions work, this section can be used to quantify your data in a quick and easy way.
- **`STEP 1`:** 🧪 **Batch process *multiple images* from a <ins>SINGLE EXPERIMENT</ins>**
- **`STEP 2`:** 🧮 **Summarize metrics *per image* across <INS>ONE OR MORE EXPERIMENTS</ins>**

---------
## **Organelle Morphology**

### summary of steps

🛠️ **BUILD FUNCTION PROTOTYPE**

- **`0`** - Apply Cell Mask *(preliminary step)*

- **`1`** - Build the list of measurements we want to include from regionprops 

- **`2`** - Add additional measurements as *"extra_properties"* with custom functions

    - define a function to retrieve the standard deviation of the region's intensity values

- **`3`** - Run regionprops and export values as a pandas dataframe

- **`4`** - Add additional measurements
    - surface area
    - surface area to volume ratio

⚙️ **EXECUTE FUNCTION PROTOTYPE**

- Define `_get_org_morphology_3D` function
- Run `_get_org_morphology_3D` function
- Compare to finalized `get_org_morphology_3D` function

-----
---------------------
## **IMPORTS AND LOAD IMAGE**
Details about the functions included in this subsection are outlined in the [`2.0_quantification_setup`](2.0_quantification_setup.ipynb) notebook. Please visit that notebook first if you are confused about any of the code included here.

In [1]:
from typing import List, Union
from pathlib import Path
import os
import time
import warnings
import tempfile

from infer_subc.core.img import *

import numpy as np
import pandas as pd
import napari
from napari.utils.notebook_display import nbscreenshot

from infer_subc.quantification.morphology import get_morphology_metrics, batch_process_org_morph, batch_org_morph_summary_stats
from infer_subc.utils.batch import list_image_files, find_segmentation_tiff_files
from infer_subc.core.file_io import read_czi_image, read_tiff_image
from infer_subc.quantification.batch import load_existing_keys_csv, append_atomic_csv

pd.set_option('display.max_columns', None)

#### &#x1F3C3; **Run code; no user input required**

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following information about your data: `raw_img_type`, `data_root_path`, `raw_data_path`, `seg_data_path`, and `quant_data_path`.

In [2]:
#### USER INPUT REQUIRED ###
raw_img_type = ".czi"
data_root_path = Path(os.path.expanduser("~")) / "Documents/Python_Scripts/Infer-subc"
raw_data_path = data_root_path / "raw_two"
seg_data_path = data_root_path / "out_two"
quant_data_path = data_root_path / "quant_two"

#### &#x1F3C3; **Run code; no user input required**

In [3]:
# Create the output directory to save the segmentation outputs in.
if not Path.exists(quant_data_path):
    Path.mkdir(quant_data_path)
    print(f"making {quant_data_path}")

# Create a list of the file paths for each image in the input folder. Select test image path.
raw_img_file_list = list_image_files(raw_data_path,raw_img_type)
pd.set_option('display.max_colwidth', None)
pd.DataFrame({"Image Name":raw_img_file_list})

,Image Name
0,C:\Users\Shannon\Documents\Python_Scripts\Infer-subc\raw_two\a24hrs-Ctrl_14_Unmixing.czi
1,C:\Users\Shannon\Documents\Python_Scripts\Infer-subc\raw_two\a48hrs-Ctrl + oleic acid_01_Unmixing.czi


#### &#x1F6D1; &#x270D; **User Input Required:**

Use the list above to specify which image you wish to analyze based on its index: `test_img_n`

In [4]:
#### USER INPUT REQUIRED ###
test_img_n = 0

#### &#x1F3C3; **Run code; no user input required**

In [5]:
# Read in the image and metadata as an ndarray and dictionary from the test image selected above. 
test_img_name = raw_img_file_list[test_img_n]
img_data,meta_dict = read_czi_image(test_img_name)

# Define some of the metadata features.
channel_names = meta_dict['name']
meta = meta_dict['metadata']['aicsimage']
scale = meta_dict['scale']
channel_axis = meta_dict['channel_axis']
file_path = meta_dict['file_name']

print("Metadata information")
print(f"File path: {file_path}")
for i in list(range(len(channel_names))):
    print(f"Channel {i} name: {channel_names[i]}")
print(f"Scale (ZYX): {scale}")
print(f"Channel axis: {channel_axis}")

Metadata information
File path: C:\Users\Shannon\Documents\Python_Scripts\Infer-subc\raw_two\a24hrs-Ctrl_14_Unmixing.czi
Channel 0 name: 0 :: a24hrs-Ctrl_14_Unmixing-0 :: Nuclei_Jan22
Channel 1 name: 0 :: a24hrs-Ctrl_14_Unmixing-0 :: Lyso+405_Jan22
Channel 2 name: 0 :: a24hrs-Ctrl_14_Unmixing-0 :: Mito+405_Jan22
Channel 3 name: 0 :: a24hrs-Ctrl_14_Unmixing-0 :: Golgi+405_Jan22
Channel 4 name: 0 :: a24hrs-Ctrl_14_Unmixing-0 :: Peroxy+405_Jan22
Channel 5 name: 0 :: a24hrs-Ctrl_14_Unmixing-0 :: ER+405_Jan22
Channel 6 name: 0 :: a24hrs-Ctrl_14_Unmixing-0 :: BODIPY+405low_Jan22
Channel 7 name: 0 :: a24hrs-Ctrl_14_Unmixing-0 :: Residuals
Scale (ZYX): (0.3891184878080979, 0.07987165184837317, 0.07987165184837318)
Channel axis: 0


#### &#x1F6D1; &#x270D; **User Input Required:**

Specify the following information about the segmentation files: - `org_file_names`, `org_channels_ordered`, `regions_file_names`, `suffix_separator`, and `mask_name`.

In [6]:
#### USER INPUT REQUIRED ###
org_file_names = ["lyso", "mito", "golgi", "perox", "ER", "LD"]
org_channels_ordered = [1, 2, 3, 4, 5, 6]
regions_file_names = ["cell", "nuc"]
suffix_separator = "-20230426_test"
mask_name = "cell"

#### &#x1F3C3; **Run code; no user input required**

In [7]:
# find file paths for segmentations
all_suffixes = org_file_names + regions_file_names
filez = find_segmentation_tiff_files(file_path, all_suffixes, seg_data_path, suffix_separator)

# read the segmentation and masks/regions files into memory
organelles = [read_tiff_image(filez[org]) for org in org_file_names]
regions = [] 
for m in regions_file_names:
    mfile = read_tiff_image(filez[m])
    regions.append(mfile)

# match the intensity channels to the segmentation files
intensities = [img_data[ch] for ch in org_channels_ordered]

# open viewer and add images
viewer = napari.Viewer()
for r, reg in enumerate(regions_file_names):
    viewer.add_image(regions[r],
                     scale=scale,
                     name=f"{reg} mask")

# colors = ["red", "bop orange", "yellow", "green", "blue", "cyan", "magenta", "bop purple"]
for o, org in enumerate(org_file_names):
    viewer.add_image(intensities[o],
                     scale=scale,
                     name=f"{org} intensity channel")
    viewer.add_labels(organelles[o],
                      scale=scale,
                      name=f"{org} segmentation")
viewer.grid.enabled = True
viewer.reset_view()

print("The following matching files were found and can now be viewed in Napari:")
filez

The following matching files were found and can now be viewed in Napari:


{'raw': WindowsPath('C:/Users/Shannon/Documents/Python_Scripts/Infer-subc/raw_two/a24hrs-Ctrl_14_Unmixing.czi'),
 'lyso': WindowsPath('C:/Users/Shannon/Documents/Python_Scripts/Infer-subc/out_two/a24hrs-Ctrl_14_Unmixing-20230426_test-lyso.tiff'),
 'mito': WindowsPath('C:/Users/Shannon/Documents/Python_Scripts/Infer-subc/out_two/a24hrs-Ctrl_14_Unmixing-20230426_test-mito.tiff'),
 'golgi': WindowsPath('C:/Users/Shannon/Documents/Python_Scripts/Infer-subc/out_two/a24hrs-Ctrl_14_Unmixing-20230426_test-golgi.tiff'),
 'perox': WindowsPath('C:/Users/Shannon/Documents/Python_Scripts/Infer-subc/out_two/a24hrs-Ctrl_14_Unmixing-20230426_test-perox.tiff'),
 'ER': WindowsPath('C:/Users/Shannon/Documents/Python_Scripts/Infer-subc/out_two/a24hrs-Ctrl_14_Unmixing-20230426_test-ER.tiff'),
 'LD': WindowsPath('C:/Users/Shannon/Documents/Python_Scripts/Infer-subc/out_two/a24hrs-Ctrl_14_Unmixing-20230426_test-LD.tiff'),
 'cell': WindowsPath('C:/Users/Shannon/Documents/Python_Scripts/Infer-subc/out_two/a24h

------
-----
## **EXPLANATION OF STEPS** <a id='explanation'></a>

-----
### 🦠 **Quantify one or more organelles from <ins>ONE IMAGE</ins>**

#### **`STEP 1` - Select mask from regions list, if applicable**

##### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** The analysis outlined below can be carried out on the entire image (`mask_name`=None), or one a single region within the image (`mask_name`={region-suffix}). For example, you may wish to ensure the analysis is carried out at the single cell level, in which case you would provide the cell segmentation file as the mask. In the cell below, the cell mask is selected from the `regions` list created above. During the analysis, any organelles outside of the cell will be ignored. 

## **Define `_get_org_morphology_3D` function**

Based on the _prototyping_ above define the function to quantify amount, size, and shape of the cell regions.

In [8]:
def _get_regions_morphology(source_file_path: str,
                           list_region_names: Union[List[str], None]=None,
                           list_region_segs: Union[List[np.ndarray], None]=None,
                           list_intensity_img: Union[List[np.ndarray], None]=None,
                           list_channel_names: Union[List[str], None]=None,
                           mask_name: Union[str, None]=None,
                           scale: Union[tuple, None]=None) -> pd.DataFrame:
    """
    Measure morphology metrics of masks/regions included in the large infer-subc pipeline (e.g. cell, nucleus, etc.).

    Parameters
    ------------
    source_file: str
        Path to the source image file. This will be used as part of the metadata information in the output table. 
        The input images are not derived from this path, but rather are provided directly as arrays in the list_obj_segs and 
        list_intensity_img variables below.
    list_region_names: Union[List[str], None]
        List of segmented region/mask names. These names should match the suffix on the segmentation image files.
        This should include:
            - a mask segmentation, such as the cell mask, for masking during all interactions analysis; else, the entire image will be 
            quantified. Only one objects per mask image will be analyzed. If there are more than one included, they will be combined 
            prior to analysis and the entire region will be quantified. If no mask is provided, the entire image will be quantified.
            - a centering object, such as the nucleus, for distribution analysis; else the center of the mask region will be used as 
            the XY distribution centering point if distribution analysis is included.
    list_region_segs: Union[List[np.ndarray], None]
        List of 3D region segmentation arrays matching the order specified in list_region_names. Specify None if no regions are provided.
    list_intensity_img: Union[List[np.ndarray], None]
        List of 3D intensity channels from the raw image. Any number of channels can be included.
        These names will be used to rename the intensity measurement columns in the output table.
        If no intensity analysis is to be included, specify None here.
    list_channel_names: Union[List[str], None]
        List of names for each intensity channel provided in list_intensity_img. The order should match the order of the channels in list_intensity_img.
    mask_name: Union[str, None]
        Name of the region to use as the mask for analysis; if not specified, the entire image will be quantified.
        The mask_name should match one of the names provided in list_region_names. This object will be used to mask
        all other objects before quantitative analysis is performed. It will also be included as one of the analyzed objects.
    scale: Union[tuple,None] = None
        a tuple that contains the real world dimensions for each dimension in the image (Z, Y, X)
            
    Returns
    -------------
    pandas dataframe of containing regionprops measurements (columns) for each object in the segmentation image (rows) and the regionprops object

    """
    # Validate inputs
    if list_region_names is None or list_region_segs is None:
        raise ValueError("You must provide both list_region_names and list_region_segs arguments.")
    if len(list_region_names) != len(list_region_segs):
        raise ValueError("The length of list_region_names must match the length of list_region_segs.")
    if list_intensity_img is None or list_channel_names is None:
        raise ValueError("You must provide both list_intensity_img and list_channel_names arguments.")
    if len(list_intensity_img) != len(list_channel_names):
        raise ValueError("The length of list_intensity_img must match the length of list_channel_names.")

    if isinstance(source_file_path, str): source_file_path = Path(source_file_path)
    print(f"Quantifying region morphology from {source_file_path.name}")

    # verify only one object per mask image; if more than one, combine them into a single object
    ## TODO: update to multi-object analysis later
    for i, region_seg in enumerate(list_region_segs):
        unique_objs = np.unique(region_seg)
        unique_objs = unique_objs[unique_objs != 0]  # exclude background
        if len(unique_objs) > 1:
            warnings.warn(f"More than one object found in region segmentation '{list_region_names[i]}'. Combining all objects into a single object for analysis.")
            combined_seg = np.isin(region_seg, unique_objs).astype(np.uint16)
            list_region_segs[i] = combined_seg

    # specify the mask image to use during quantification
    if list_region_names is None or list_region_segs is None:
        print("No regions provided. No mask will be applied before analysis.")
        mask = None
    elif mask_name is None or mask_name not in list_region_names:
        if mask_name is not None:
            raise ValueError(f"Mask '{mask_name}' not found. No mask will be applied before analysis.")
        mask = None
        mask_name = None
    else:
        mask = list_region_segs[list_region_names.index(mask_name)]

    # merge intensity images to create a single np.ndarray
    if list_intensity_img is None:
        intensity_img = None
        print("No intensity images provided. Morphology metrics that require intensity images will not be calculated.")
    else:
        intensity_img = np.stack(list_intensity_img, axis=0)

    # empty list to collect a morphology data for each organelle
    regions_tab = []

    # loop through the list of organelles and run the get_morphology_metrics function
    for j, target in enumerate(list_region_names):
        region_seg = list_region_segs[j]

        # run get_morphology_metrics function to output a table of measurements
        region_metrics = get_morphology_metrics(segmentation_img=region_seg, 
                                            seg_name=target,
                                            intensity_img=intensity_img, 
                                            intensity_ch_names=list_channel_names,
                                            channel_axis=0, # default to 0 because intensities are merged from list on axis 0
                                            mask=mask,
                                            mask_name=mask_name,
                                            scale=scale)
        
        # add table to list above
        regions_tab.append(region_metrics)

    # combine the lists for each organelle into one table
    final_region_tab = pd.concat(regions_tab, ignore_index=True)

    # add a new column to list the name of the image these data are derived from 
    final_region_tab.insert(loc=0,column='image_name',value=source_file_path.stem)

    return final_region_tab

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block applies the function above to your test image. The settings specified above are applied here.

In [9]:
# test function above
region_morph_tab = _get_regions_morphology(source_file_path = file_path,
                                          list_region_names = regions_file_names,
                                          list_region_segs = regions,
                                          list_intensity_img = intensities,
                                          list_channel_names = org_file_names,
                                          mask_name=mask_name,
                                          scale=scale)
display(region_morph_tab)

# check is function output matches previous output
# print(f"\nThe output here matches the output from the individual steps above: {region_morph_tab.equals(final_region_tab)}")

Quantifying region morphology from a24hrs-Ctrl_14_Unmixing.czi


C:\Users\Shannon\AppData\Local\Temp\ipykernel_35304\2523065191.py:64: UserWarning: More than one object found in region segmentation 'nuc'. Combining all objects into a single object for analysis.
  warnings.warn(f"More than one object found in region segmentation '{list_region_names[i]}'. Combining all objects into a single object for analysis.")


Warning(s) suppressed while quantifying cell. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying nuc. See 'method_morphology.ipynb' notebook for more details.


,image_name,mask_name,scale,object,label,centroid-0,centroid-1,centroid-2,bbox-0,bbox-1,bbox-2,bbox-3,bbox-4,bbox-5,volume,surface_area,SA_to_volume_ratio,equivalent_diameter,extent,euler_number,solidity,axis_major_length,min_intensity-lyso-ch,min_intensity-mito-ch,min_intensity-golgi-ch,min_intensity-perox-ch,min_intensity-ER-ch,min_intensity-LD-ch,max_intensity-lyso-ch,max_intensity-mito-ch,max_intensity-golgi-ch,max_intensity-perox-ch,max_intensity-ER-ch,max_intensity-LD-ch,mean_intensity-lyso-ch,mean_intensity-mito-ch,mean_intensity-golgi-ch,mean_intensity-perox-ch,mean_intensity-ER-ch,mean_intensity-LD-ch,standard_deviation_intensity-lyso-ch,standard_deviation_intensity-mito-ch,standard_deviation_intensity-golgi-ch,standard_deviation_intensity-perox-ch,standard_deviation_intensity-ER-ch,standard_deviation_intensity-LD-ch,cell_volume
0,a24hrs-Ctrl_14_Unmixing,cell,"(0.3891, 0.0799, 0.0799)",cell,1,2.926375,29.817285,23.944586,0,0,0,16,659,592,3835.846084,2608.725791,0.680091,19.421712,0.247552,-13,0.626284,54.543657,0.0,0.0,0.0,0.0,0.0,0.0,25817.0,43784.0,65535.0,25484.0,45278.0,43756.0,438.778382,1103.154897,968.052221,686.640820,2262.953316,615.328306,1176.228311,3162.565822,3291.795243,1287.589652,2878.302425,1163.771714,3835.846084
1,a24hrs-Ctrl_14_Unmixing,cell,"(0.3891, 0.0799, 0.0799)",nuc,1,2.628936,31.602057,28.150081,0,284,242,16,504,463,1037.508176,650.256734,0.626749,12.560231,0.537266,1,0.913476,21.135161,0.0,0.0,0.0,0.0,0.0,0.0,20433.0,15222.0,15249.0,15353.0,19653.0,7637.0,367.603307,159.087391,236.961146,473.744757,1447.220691,357.386144,652.078905,453.954104,496.541180,887.180795,1842.257956,442.751818,3835.846084


##### &#x1F453; **FYI:** This function has been added to `infer_subc.quantification.regions` and can be imported with the following:
> ```python
> from infer_subc.quantification.regions import get_regions_morphology
> ```

## <mark> NEED TO CHECK STILL

In [10]:
from infer_subc.quantification.regions import get_regions_morphology

# test function above
region_morph_tab = get_regions_morphology(source_file_path = file_path,
                                          list_region_names = regions_file_names,
                                          list_region_segs = regions,
                                          list_intensity_img = intensities,
                                          list_channel_names = org_file_names,
                                          mask_name=mask_name,
                                          scale=scale)
display(region_morph_tab)

# check is function output matches previous output
# print(f"\nThe output here matches the output from the individual steps above: {region_morph_tab.equals(final_region_tab)}")

ImportError: cannot import name 'get_regions_morphology' from 'infer_subc.quantification.regions' (C:\Users\Shannon\Documents\Python_Scripts\SCohen infer-subc\infer-subc\infer_subc\quantification\regions.py)

-----
### 🧪 **Batch process *multiple images* from a <ins>SINGLE EXPERIMENT</ins>**

In [11]:
def _batch_process_regions_morph(dataset_name: str,
                                 raw_path: Union[Path,str], 
                                 seg_path: Union[Path,str],
                                 quant_path: Union[Path, str], 
                                 raw_file_type: str,
                                 channel_axis: Union[int, None]=None,
                                 channel_names: Union[List[int], None]=None,
                                 region_names: Union[List[str], None]=None,
                                 mask_name: Union[str, None]=None,
                                 use_scale: bool=True,
                                 seg_suffix: Union[str, None]=None):
    """  
    batch process quantification of the regions morphology for a single dataset. This function is currently 
    optimized to process images from one file folder per image type (e.g., raw, segmentation) the output csv 
    files are saved to the indicated quant_path folder.

    Parameters:
    ----------
    dataset_name : str
        A unique string identifier for the dataset being processed. It will be included as metadata in output tables and as 
        part of the output files names. It will be used to identify if any data has already been collected for this dataset.
    raw_path: Union[Path,str]
        Path or str to the folder that contains the raw image files
    seg_path: Union[Path,str]
        Path or str to the folder that contains the segmentation tiff files
    quant_path: Union[Path, str]
        Path or str to the folder that the output datatables will be saved to
    raw_file_type: str
        File type of the raw images (e.g., "czi", "tiff")
    channel_axis : int
        Axis corresponding to the channels in the image data
    channel_names: List[str]
        List of channel names associated to each channel in the raw image data; if you wish to exclude a particular channel
        from the intensity analysis, write None instead of the channel name.
    region_names: Union[List[str], None]=None
        List of region names to analyze. Usually ['cell', 'nuc'] for cell mask and nucleus.
        If no regions are to be included, specify None here.
    mask_name: Union[str, None]=None
        Name of the region to use for segmentation (if any). This name should be included in the regions_name variable.
        If None, the entire image will be quantified.
    use_scale: bool=True
        Whether to apply scaling to the quantitative data; scaled data will be in real world units (e.g., microns) rather than pixels/voxels
    seg_suffix:Union[str, None]=None
        Any additional text that is included in the segmentation tiff files between the file stem and the segmentation suffix, not including the initial "-"

    Returns:
    ----------
    None: files are saved to quant_path directly
    """
    
    start = time.time()
    count = 0

    # create path objects if inputs are strings
    if isinstance(raw_path, str): raw_path = Path(raw_path)
    if isinstance(seg_path, str): seg_path = Path(seg_path)
    if isinstance(quant_path, str): quant_path = Path(quant_path)
    
    # create directory is it doesn't exist
    if not Path.exists(quant_path):
        Path.mkdir(quant_path)
        print(f"Output file path not found. Making {quant_path}.")

    # check if any existing data is present in outfiles to skip already processed images
    unique_keys = ['dataset', 'image_name']

    regions_path = quant_path / f"{dataset_name}_regions_morphology_metrics.csv"
    existing_morpho_keys = load_existing_keys_csv(regions_path, unique_keys)

    # reading list of files from the raw path
    img_file_list = list_image_files(raw_path, raw_file_type)
    len_file_list = len(img_file_list)

    # loop through list of cell analyzing each and appending the data to the empty list
    for img_f in img_file_list:
        img_start = time.time()
        count = count + 1
        # skip files that have already been processed
        if (dataset_name, img_f.stem) in existing_morpho_keys:
            print(f"Skipping {img_f.name} as it is already listed in the output file(s).")
            continue
        # process analysis for this cells
        else:
            filez = find_segmentation_tiff_files(img_f, region_names, seg_path, seg_suffix)

            # read in raw file and metadata
            img_data, meta_dict = read_czi_image(filez["raw"])

            # create intensities from raw file as list baseed on channel_name list
            if channel_names is None:
                intensities = None
                print("No intensity channel information provided.")
            else:
                if channel_axis != 0:
                    img_data = np.moveaxis(img_data, channel_axis, 0)
                intensities = [img_data[i] for i, ch in enumerate(channel_names) if ch is not None]
                channel_names = [ch for ch in channel_names if ch is not None]

            # store organelle images as list
            regions = [read_tiff_image(filez[org]) for org in region_names]

            # define the scale
            if use_scale is True:
                scale = meta_dict['scale']
            else:
                scale = None

            regions_metrics = _get_regions_morphology(source_file_path=img_f,
                                                  list_region_names=region_names,
                                                  list_region_segs=regions,
                                                  list_intensity_img=intensities,
                                                  list_channel_names=channel_names,
                                                  mask_name=mask_name,
                                                  scale=scale)
            
            # save the morphology table data per image directly to csv
            regions_metrics.insert(loc=0,column='dataset',value=dataset_name)
            append_atomic_csv(regions_path, regions_metrics)
            del regions_metrics  # free up memory

            end2 = time.time()
            print(f"Completed quantification of {meta_dict['file_name']} in {(end2-img_start)/60} mins.")
            print(f"{count}/{len_file_list} images have been processed.")
            print(f"Time elapsed: {(end2-img_start)/60} mins")

    end = time.time()
    print(f"Quantification for {count} files is COMPLETE! Files saved to '{quant_path}'.")
    print(f"It took {(end - start)/60} minutes to quantify these files.")

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block applies the function above to your test image. The settings specified above are applied here.

In [12]:
use_scale = True

In [ ]:
# define a new dataset name for comparison
dataset_name2 = "test_dataset_1-comparison"

_batch_process_regions_morph(dataset_name=dataset_name2,
                        seg_path = seg_data_path,
                        quant_path = quant_data_path, 
                        raw_path = raw_data_path, 
                        raw_file_type = raw_img_type,
                        channel_axis = channel_axis,
                        channel_names = org_file_names,
                        region_names = regions_file_names,
                        mask_name = mask_name,
                        use_scale = use_scale,
                        seg_suffix = suffix_separator)

# # compare the output files from both dataset runs to ensure they are identical
# morpho_file1 = quant_data_path / f"{dataset_name}_org_morphology_metrics.csv"
# morpho_file2 = quant_data_path / f"{dataset_name2}_org_morphology_metrics.csv"


# # determine if files are identical except for the dataset name column
# if morpho_file1.exists() and morpho_file2.exists():
#     morph_1 = pd.read_csv(morpho_file1)
#     morph_2 = pd.read_csv(morpho_file2)
#     morph_2['dataset'] = morph_1['dataset']  # set dataset names to be the same for comparison
#     print("\nMorphology metrics files identical:", morph_1.equals(morph_2))

Quantifying region morphology from a24hrs-Ctrl_14_Unmixing.czi
Warning(s) suppressed while quantifying cell. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying nuc. See 'method_morphology.ipynb' notebook for more details.
Completed quantification of C:\Users\Shannon\Documents\Python_Scripts\Infer-subc\raw_two\a24hrs-Ctrl_14_Unmixing.czi in 1.0514066060384115 mins.
1/2 images have been processed.
Time elapsed: 1.0514066060384115 mins
Quantifying region morphology from a48hrs-Ctrl + oleic acid_01_Unmixing.czi
Warning(s) suppressed while quantifying cell. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying nuc. See 'method_morphology.ipynb' notebook for more details.
Completed quantification of C:\Users\Shannon\Documents\Python_Scripts\Infer-subc\raw_two\a48hrs-Ctrl + oleic acid_01_Unmixing.czi in 0.42551478147506716 mins.
2/2 images have been processed.
Time elapsed: 0.42551478147506716 mins
Quanti

,dataset,image_name,mask_name,scale,object,label,centroid-0,centroid-1,centroid-2,bbox-0,bbox-1,bbox-2,bbox-3,bbox-4,bbox-5,volume,surface_area,SA_to_volume_ratio,equivalent_diameter,extent,euler_number,solidity,axis_major_length,min_intensity-lyso-ch,min_intensity-mito-ch,min_intensity-golgi-ch,min_intensity-perox-ch,min_intensity-ER-ch,min_intensity-LD-ch,max_intensity-lyso-ch,max_intensity-mito-ch,max_intensity-golgi-ch,max_intensity-perox-ch,max_intensity-ER-ch,max_intensity-LD-ch,mean_intensity-lyso-ch,mean_intensity-mito-ch,mean_intensity-golgi-ch,mean_intensity-perox-ch,mean_intensity-ER-ch,mean_intensity-LD-ch,standard_deviation_intensity-lyso-ch,standard_deviation_intensity-mito-ch,standard_deviation_intensity-golgi-ch,standard_deviation_intensity-perox-ch,standard_deviation_intensity-ER-ch,standard_deviation_intensity-LD-ch,cell_volume
0,test_dataset_1-comparison,a24hrs-Ctrl_14_Unmixing,cell,"(0.3891, 0.0799, 0.0799)",lyso,1,0.000000,21.855788,15.930764,0,271,197,1,278,202,0.054612,0.698442,12.789117,0.470721,0.628571,1,inf,0.636832,0.0,0.0,0.0,0.0,0.0,69.0,6740.0,5202.0,3117.0,2018.0,2983.0,2227.0,2750.772727,1882.772727,594.227273,590.636364,890.727273,915.227273,1982.010364,1447.342327,977.096810,747.016949,834.106391,621.463224,3835.846084
1,test_dataset_1-comparison,a24hrs-Ctrl_14_Unmixing,cell,"(0.3891, 0.0799, 0.0799)",lyso,2,0.000000,22.349087,19.423787,0,278,241,1,283,246,0.039718,0.561866,14.146382,0.423314,0.640000,1,inf,0.592344,174.0,0.0,0.0,0.0,0.0,0.0,4247.0,3179.0,1842.0,2080.0,2788.0,1090.0,2239.250000,824.750000,124.062500,626.437500,1537.625000,436.187500,1050.838980,1071.536659,444.912136,723.195165,773.425891,288.851436,3835.846084
2,test_dataset_1-comparison,a24hrs-Ctrl_14_Unmixing,cell,"(0.3891, 0.0799, 0.0799)",lyso,3,0.000000,22.670667,16.512820,0,279,203,1,289,210,0.076954,1.025384,13.324701,0.527728,0.442857,1,inf,0.792587,315.0,0.0,0.0,0.0,0.0,137.0,4582.0,10680.0,6173.0,3774.0,4225.0,3778.0,2201.580645,5704.387097,1152.935484,857.225806,863.870968,1393.451613,1087.198583,2358.647790,1798.610628,1154.611252,961.748030,871.031011,3835.846084
3,test_dataset_1-comparison,a24hrs-Ctrl_14_Unmixing,cell,"(0.3891, 0.0799, 0.0799)",lyso,4,0.190035,28.081387,22.904589,0,348,283,2,357,290,0.106742,1.638994,15.354710,0.588544,0.341270,1,0.704918,0.924011,0.0,0.0,0.0,0.0,0.0,0.0,6208.0,8219.0,4957.0,5493.0,5930.0,2757.0,2170.000000,1038.860465,654.000000,1257.697674,2561.860465,974.627907,1508.550745,1523.534287,1096.552165,1509.426127,1545.488055,774.551274,3835.846084
4,test_dataset_1-comparison,a24hrs-Ctrl_14_Unmixing,cell,"(0.3891, 0.0799, 0.0799)",lyso,5,0.440619,28.385563,24.257490,0,352,299,3,360,310,0.337603,3.146287,9.319491,0.863911,0.515152,1,0.839506,1.333907,0.0,0.0,0.0,0.0,0.0,0.0,8111.0,3941.0,4887.0,5959.0,8111.0,3328.0,2645.058824,361.911765,497.176471,835.441176,2841.661765,694.889706,1895.408771,757.711284,961.572600,1278.925743,1656.294195,738.690143,3835.846084
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
324,test_dataset_1-comparison,a48hrs-Ctrl + oleic acid_01_Unmixing,cell,"(0.3891, 0.0799, 0.0799)",LD,15,2.334711,45.420346,29.521450,6,567,367,7,572,373,0.044683,1.263401,28.274919,0.440265,0.600000,1,inf,0.655341,0.0,0.0,0.0,0.0,0.0,3132.0,40.0,1027.0,1421.0,783.0,601.0,8665.0,2.222222,343.611111,303.500000,212.777778,189.000000,6092.611111,9.162457,319.825324,386.700609,291.077797,199.237993,1622.816795,2302.843664
325,test_dataset_1-comparison,a48hrs-Ctrl + oleic acid_01_Unmixing,cell,"(0.3891, 0.0799, 0.0799)",LD,22,3.502066,37.290077,19.463723,9,465,242,10,470,247,0.039718,1.092028,27.494555,0.423314,0.640000,1,inf,0.420816,0.0,0.0,0.0,0.0,0.0,4455.0,0.0,14.0,318.0,1283.0,2020.0,9287.0,0.000000,0.875000,30.312500,225.187500,698.312500,6694.187500,0.000000,3.388860,78.578558,330.649402,524.511644,1300.366449,2302.843664
326,tes

In [ ]:
morpho_file2 = quant_data_path / f"{dataset_name2}_regions_morphology_metrics.csv"
morph_2 = pd.read_csv(morpho_file2)
display(morph_2)

,dataset,image_name,mask_name,scale,object,label,centroid-0,centroid-1,centroid-2,bbox-0,bbox-1,bbox-2,bbox-3,bbox-4,bbox-5,volume,surface_area,SA_to_volume_ratio,equivalent_diameter,extent,euler_number,solidity,axis_major_length,min_intensity-lyso-ch,min_intensity-mito-ch,min_intensity-golgi-ch,min_intensity-perox-ch,min_intensity-ER-ch,min_intensity-LD-ch,max_intensity-lyso-ch,max_intensity-mito-ch,max_intensity-golgi-ch,max_intensity-perox-ch,max_intensity-ER-ch,max_intensity-LD-ch,mean_intensity-lyso-ch,mean_intensity-mito-ch,mean_intensity-golgi-ch,mean_intensity-perox-ch,mean_intensity-ER-ch,mean_intensity-LD-ch,standard_deviation_intensity-lyso-ch,standard_deviation_intensity-mito-ch,standard_deviation_intensity-golgi-ch,standard_deviation_intensity-perox-ch,standard_deviation_intensity-ER-ch,standard_deviation_intensity-LD-ch,cell_volume
0,test_dataset_1-comparison,a24hrs-Ctrl_14_Unmixing,cell,"(0.3891, 0.0799, 0.0799)",cell,1,2.926375,29.817285,23.944586,0,0,0,16,659,592,3835.846084,2608.725791,0.680091,19.421712,0.247552,-13,0.626284,54.543657,0.0,0.0,0.0,0.0,0.0,0.0,8392.0,25817.0,43784.0,65535.0,25484.0,45278.0,779.060937,438.778382,1103.154897,968.052221,686.640820,2262.953316,968.509916,1176.228311,3162.565822,3291.795243,1287.589652,2878.302425,3835.846084
1,test_dataset_1-comparison,a24hrs-Ctrl_14_Unmixing,cell,"(0.3891, 0.0799, 0.0799)",nuc,1,2.628936,31.602057,28.150081,0,284,242,16,504,463,1037.508176,650.256734,0.626749,12.560231,0.537266,1,0.913476,21.135161,0.0,0.0,0.0,0.0,0.0,0.0,8392.0,20433.0,15222.0,15249.0,15353.0,19653.0,2012.255031,367.603307,159.087391,236.961146,473.744757,1447.220691,1048.054565,652.078905,453.954104,496.541180,887.180795,1842.257956,3835.846084
2,test_dataset_1-comparison,a48hrs-Ctrl + oleic acid_01_Unmixing,cell,"(0.3891, 0.0799, 0.0799)",cell,1,3.647372,25.584379,30.481331,6,0,144,15,644,641,2302.843664,2435.898850,1.057779,16.384076,0.322042,-1,0.676884,44.533387,0.0,0.0,0.0,0.0,0.0,0.0,5870.0,8653.0,25938.0,31102.0,65535.0,45830.0,725.209150,142.200288,668.529139,370.048711,1134.492490,3268.545680,924.774745,375.903717,1805.859669,1255.292623,2344.635836,3919.979772,2302.843664
3,test_dataset_1-comparison,a48hrs-Ctrl + oleic acid_01_Unmixing,cell,"(0.3891, 0.0799, 0.0799)",nuc,1,3.453518,21.589952,32.064531,6,186,261,15,374,522,464.759973,684.917229,1.473701,9.610442,0.423956,1,0.708524,22.507211,0.0,0.0,0.0,0.0,0.0,0.0,5870.0,4078.0,3266.0,3270.0,6764.0,8888.0,2010.659339,192.266269,55.802654,92.418589,499.112646,814.923904,769.229406,387.811935,168.660437,229.194460,612.056468,789.859886,2302.843664


## <mark> NEED TO UPDATE AND CHECK STILL

##### &#x1F453; **FYI:** This function has been added to `infer_subc.quantification.morphology` and can be imported with the following:
> ```python
> from infer_subc.quantification.morphology import batch_process_org_morph
> ```

In [ ]:
from infer_subc.quantification.morphology import batch_process_org_morph

# define a new dataset name for comparison
dataset_name3 = "test_dataset_1-testcomparison"

batch_process_org_morph(dataset_name=dataset_name3,
                        seg_path = seg_data_path,
                        quant_path = quant_data_path, 
                        raw_path = raw_data_path, 
                        raw_file_type = raw_img_type,
                        channel_axis=channel_axis,
                        organelle_names = org_file_names,
                        organelle_channels = org_channels_ordered,
                        region_names = regions_file_names,
                        mask_name = mask_name,
                        use_scale = use_scale,
                        seg_suffix = suffix_separator)

# compare the output files from both dataset runs to ensure they are identical
morpho_file1 = quant_data_path / f"{dataset_name}_org_morphology_metrics.csv"
morpho_file3 = quant_data_path / f"{dataset_name3}_org_morphology_metrics.csv"

# determine if files are identical except for the dataset name column
if morpho_file1.exists() and morpho_file3.exists():
    morph_1 = pd.read_csv(morpho_file1)
    morph_3 = pd.read_csv(morpho_file3)
    morph_3['dataset'] = morph_1['dataset']  # set dataset names to be the same for comparison
    print("\nMorphology metrics files identical:", morph_1.equals(morph_3))

Skipping a24hrs-Ctrl_14_Unmixing.czi as it is already listed in the output file(s).
Quantifying organelle morphology from a48hrs-Ctrl + oleic acid_01_Unmixing.czi
Warning(s) suppressed while quantifying lyso. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying mito. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying golgi. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying perox. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying ER. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying LD. See 'method_morphology.ipynb' notebook for more details.
Completed quantification of C:\Users\Shannon\Documents\Python_Scripts\Infer-subc\raw_two\a48hrs-Ctrl + oleic acid_01_Unmixing.czi in 0.3212791283925374 mins.
2/2 images have been processed.
Time elapsed: 0.321279128392537

-----
### 🧮 **Summarize metrics *per image/mask* across <INS>ONE OR MORE EXPERIMENTS</ins>**

In [ ]:
def _batch_regions_morph_summary_stats(csv_path_list: List[str],
                                        out_path: str,
                                        out_prefix: str,
                                        region_names: List[str]):
    """" 
    csv_path_list: List[str],
        A list of path strings where .csv files to analyze are located.
    out_path: str,
        A path string where the summary data file will be output to
    out_prefix: str
        The prefix used to name the output file. An "_" will be included between this prefix and the file suffix.
    region_names: List[str],
        A list of region names used in the region morphology quantification (batch_process_region_morph function)
    """
    # for keeping track of dataset and file numbers
    ds_count = 0
    fl_count = 0

    ###################
    # Read in the csv files and combine them into one of each type
    ###################
    # create empty list to hold the morphology tables from different experiments
    regions_tab = []

    # loop through all of the locations listed above and find the _regions_morph files; append them to the list above
    for loc in csv_path_list:
        # list all csv files in the location
        files_store = sorted(loc.glob("*.csv"))

        # find the unique datasets in this location based on the prefixes before "_regions_morphology_metrics"
        prefixes = set(f.name.split("_regions_morphology_metrics")[0] for f in files_store if "_regions_morphology_metrics" in f.name)
        for prefix in prefixes:
            ds_count += 1
            # select only the files from this dataset
            files_subset = [f for f in files_store if f.name.startswith(prefix +"_regions_morphology_metrics")]
            for file in files_subset:
                fl_count += 1
                stem = file.stem
                if "_regions_morph" in stem:
                    test_regions = pd.read_csv(file, index_col=0)
                    regions_tab.append(test_regions)

    # combine the regions_morph lists found above into one table
    regions_df = pd.concat(regions_tab,axis=0, join='outer').reset_index()
    print(f"Found {fl_count} files from {ds_count} dataset(s) across {len(csv_path_list)} location(s).")


    ###################
    # summary stat group
    ###################
    group_by = ['dataset', 'image_name', 'mask_name', 'scale', 'object']
    sharedcolumns = ["SA_to_volume_ratio", "equivalent_diameter", "extent", "euler_number", "solidity", "axis_major_length"]  + list(regions_df.filter(regex=".*intensity.*").columns)
    ag_func_standard = ['mean', 'median', 'std']

    ###################
    # summarize morphology metrics
    ###################
    tab1 = regions_df[group_by + ['label']].groupby(group_by).agg(['count'])
    tab1.rename(columns={'label': 'region'}, inplace=True)
    tab2 = regions_df[group_by + ['volume', 'surface_area']].groupby(group_by).agg(['sum'] + ag_func_standard)
    tab3 = regions_df[group_by + sharedcolumns].groupby(group_by).agg(ag_func_standard)
    regions_summary = pd.merge(tab1, tab2, 'outer', on=group_by)
    regions_summary = pd.merge(regions_summary, tab3, 'outer', on=group_by)

    # Get mask_name and corresponding volume column per group & calculate volume fraction
    mask_names = regions_df.groupby(group_by)['mask_name'].first()
    mask_volume_data = regions_df.groupby(group_by).first().apply(lambda row: row[f"{mask_names.loc[row.name]}_volume"], axis=1)
    regions_summary.insert(regions_summary.columns.get_loc(('volume', 'sum')) + 1, ('volume', 'fraction'), regions_summary[('volume', 'sum')]/mask_volume_data)

    #######################
    # fill gaps & NA values
    #######################
    # Ensure all possible interactions are represented (if missing fill with NaN)
    for ind in regions_summary.index.droplevel(4).unique().to_list():
        for row in region_names:
            if ind+(row,) not in regions_summary.index:
                regions_summary.loc[ind+(row,)] = np.nan

    # fill NA with 0 for specific columns
    fill_dict = {('region', 'count'): 0, 
                ('volume', 'sum'): 0,
                ('surface_area', 'sum'): 0,
                ('volume', 'fraction'): 0}
    regions_summary = regions_summary.fillna(value=fill_dict)

    # if (region, count) is 1, set mean, median, and std to NaN
    single_site_mask = regions_summary[('region', 'count')] == 1
    for col in sharedcolumns+['volume', 'surface_area']:
        regions_summary.loc[single_site_mask, (col, 'std')] = np.nan

    regions_summary.sort_index(inplace=True)

    ###################
    # flatten datasheet and export
    ###################
    # export before unstacking
    if (Path(out_path) / f"{out_prefix}_per_region_morphology_summarystats.csv").exists():
        raise FileExistsError(f"CAUTION: {out_prefix}_per_region_morphology_summarystats.csv already exists and will not be overwritten. Move the existing file, change the `out_prefix` or `quant_data_path` to continue without error.")
    else:
        regions_summary.to_csv(str(out_path) + f"/{out_prefix}_per_region_morphology_summarystats.csv", mode='x')
        print(f"Exported per-region morphology summary statistics (before unstacking) to {quant_data_path}/{out_prefix}_per_region_morphology_summarystats.csv")
    regions_morph_final = regions_summary.unstack(-1)
    regions_morph_final.columns = ["_".join((col_name[1], col_name[-1], col_name[0])) for col_name in regions_morph_final.columns.to_flat_index()]
    regions_morph_final.columns = [col.replace('sum', 'total') for col in regions_morph_final.columns]
    regions_morph_final.columns = [col.replace(col, 'mask_volume') if 'mask' in col else col for col in regions_morph_final.columns]
    regions_morph_final = regions_morph_final.loc[:, ~regions_morph_final.columns.duplicated()]
    regions_morph_final.reset_index(inplace=True)

    ###################
    # export summary sheets
    ###################
    if (Path(out_path) / f"{out_prefix}_regions_morphology_summarystats.csv").exists():
        raise FileExistsError(f"CAUTION: {out_prefix}_regions_morphology_summarystats.csv already exists and will not be overwritten. Move the existing file, change the `out_prefix` or `quant_data_path` to continue without error.")
    else:
        regions_morph_final.to_csv(str(out_path) + f"/{out_prefix}_regions_morphology_summarystats.csv", mode='x')
        print(f"Exported regions morphology summary statistics (after unstacking) to {out_path}/{out_prefix}_regions_morphology_summarystats.csv")
    print(f"Regions morphology summary is complete.")
    return regions_summary

In [19]:
csv_path_list = [quant_data_path]

In [20]:
# define new out_prefix for comparison
out_prefix2 = "all_datasets_1-comparison"

# test the function above
test_regions_summary = _batch_regions_morph_summary_stats(csv_path_list = csv_path_list,
                                                            out_path = quant_data_path,
                                                            out_prefix = out_prefix2,
                                                            region_names = regions_file_names)

# compare the output files from both dataset runs to ensure they are identical
# morpho_file1 = quant_data_path / f"{out_prefix}_regions_morphology_summarystats.csv"
morpho_file2 = quant_data_path / f"{out_prefix2}_regions_morphology_summarystats.csv"
# determine if files are identical except for the dataset name column
# if morpho_file1.exists() and morpho_file2.exists():
    # morpho_1 = pd.read_csv(morpho_file1)
morpho_2 = pd.read_csv(morpho_file2)
    # print("\nMorphology summarystats files identical:", morpho_1.equals(morpho_2))

Found 1 files from 1 dataset(s) across 1 location(s).
Exported per-region morphology summary statistics (before unstacking) to C:\Users\Shannon\Documents\Python_Scripts\Infer-subc\quant_two/all_datasets_1-comparison_per_region_morphology_summarystats.csv
Exported regions morphology summary statistics (after unstacking) to C:\Users\Shannon\Documents\Python_Scripts\Infer-subc\quant_two/all_datasets_1-comparison_regions_morphology_summarystats.csv
Regions morphology summary is complete.


In [21]:
test_regions_summary

region  \
                                                                                                          count   
dataset                   image_name                           mask_name scale                    object          
test_dataset_1-comparison a24hrs-Ctrl_14_Unmixing              cell      (0.3891, 0.0799, 0.0799) cell        1   
                                                                                                  nuc         1   
                          a48hrs-Ctrl + oleic acid_01_Unmixing cell      (0.3891, 0.0799, 0.0799) cell        1   
                                                                                                  nuc         1   

                                                                                                               volume  \
                                                                                                                  sum   
dataset                   image_name                           mask_name scale                    object                
test_dataset_1-comparison a24hrs-Ctrl_14_Unmixing              cell      (0.3891, 0.0799, 0.0799) cell    3835.846084   
                                                                                                  nuc     1037.508176   
                          a48hrs-Ctrl + oleic acid_01_Unmixing cell      (0.3891, 0.0799, 0.0799) cell    2302.843664   
                                                                                                  nuc      464.759973   

                                                                                                                    \
                                                                                                          fraction   
dataset                   image_name                           mask_name scale                    object             
test_dataset_1-comparison a24hrs-Ctrl_14_Unmixing              cell      (0.3891, 0.0799, 0.0799) cell    1.000000   
                                                                                                  nuc     0.270477   
                          a48hrs-Ctrl + oleic acid_01_Unmixing cell      (0.3891, 0.0799, 0.0799) cell    1.000000   
                                                                                                  nuc     0.201820   

                                                                                                                       \
                                                                                                                 mean   
dataset                   image_name                           mask_name scale                    object                
test_dataset_1-comparison a24hrs-Ctrl_14_Unmixing              cell      (0.3891, 0.0799, 0.0799) cell    3835.846084   
                                                                                                  nuc     1037.508176   
                          a48hrs-Ctrl + oleic acid_01_Unmixing cell      (0.3891, 0.0799, 0.0799) cell    2302.843664   
                                                                                                  nuc      464.759973   

                                                                                                                       \
                                                                                                               median   
dataset                   image_name                           mask_name scale                    object                
test_dataset_1-comparison a24hrs-Ctrl_14_Unmixing              cell      (0.3891, 0.0799, 0.0799) cell    3835.846084   
                                                                                                  nuc     1037.508176   
                          a48hrs-Ctrl + oleic acid_01_Unmixing cell      (0.3891, 0.0799, 0.0799) cell    2302.843664   
                                                  